# CLIP4CAD-GFA v4.8.2 Training

**Optimized Simplified Topology-Aware Training**

## Key Changes from v4.8.1
- **25% Capacity Increase**: d=320, d_proj=160, more codes
- **Smooth Curriculum**: Gradual transitions between loss components
- **Temperature Annealing**: tau: 0.07 → 0.05
- **Warmup Scheduler**: Linear warmup + cosine decay
- **Gradient Checkpointing**: ~40% memory reduction

## Model Configuration
| Parameter | v4.8.1 | v4.8.2 | Rationale |
|-----------|--------|--------|----------|
| d | 256 | 320 | +25% capacity |
| d_proj | 128 | 160 | Maintain ratio |
| n_category | 16 | 20 | More semantic codes |
| n_type_per_cat | 8 | 10 | Finer granularity |
| n_spatial | 16 | 20 | Better spatial encoding |
| num_brep_tf_layers | 4 | 5 | +1 layer depth |
| num_heads | 8 | 10 | Keep head_dim=32 |

**Result:** ~15M params (up from ~12M), ~1040 codes (up from 672)

## Training Stages
| Stage | Epochs | LR | Warmup | Goal |
|-------|--------|-----|--------|------|
| 0 | 12 | 3e-4 | 1 epoch | Anchor BRep to PC |
| 1 | 15 | 1e-4 | 2 epochs | Text + Codebook |
| 2 | 10 | 2e-5 | 1 epoch | Gap Closing |

In [1]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Clean memory
gc.collect()
torch.cuda.empty_cache()

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4090
Memory: 25.8 GB


In [21]:
# Cell 2: Configuration
# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS
# ═══════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"
ALIGNED_DIR = DATA_ROOT / "aligned"

# PC and Text files
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

# Output
OUTPUT_DIR = Path("../outputs/gfa_v4_8_2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS (v4.8.2 Optimized)
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 256
# Stage epochs (adjusted for stability)
STAGE0_EPOCHS = 12    # Anchor BRep to PC
STAGE1_EPOCHS = 15    # Add text + codebook
STAGE2_EPOCHS = 10    # Close gap + hard negatives
TOTAL_EPOCHS = STAGE0_EPOCHS + STAGE1_EPOCHS + STAGE2_EPOCHS  # 37 total

# Learning rates per stage (with warmup)
STAGE0_LR = 3e-4      # Higher for initial learning
STAGE1_LR = 1e-4      # Standard
STAGE2_LR = 2e-5      # Lower for fine-tuning

# Warmup epochs per stage
STAGE0_WARMUP = 1
STAGE1_WARMUP = 2
STAGE2_WARMUP = 1

# Weight decay
WEIGHT_DECAY = 0.01

MAX_GRAD_NORM = 1.0

print("Configuration (v4.8.2 Optimized):")
print(f"  Data root: {DATA_ROOT}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Stage 0: {STAGE0_EPOCHS} epochs, LR={STAGE0_LR}, warmup={STAGE0_WARMUP}")
print(f"  Stage 1: {STAGE1_EPOCHS} epochs, LR={STAGE1_LR}, warmup={STAGE1_WARMUP}")
print(f"  Stage 2: {STAGE2_EPOCHS} epochs, LR={STAGE2_LR}, warmup={STAGE2_WARMUP}")
print(f"  Total: {TOTAL_EPOCHS} epochs")
print(f"  Output: {OUTPUT_DIR}")

Configuration (v4.8.2 Optimized):
  Data root: d:\Defect_Det\MMCAD\data
  Batch size: 256
  Stage 0: 12 epochs, LR=0.0003, warmup=1
  Stage 1: 15 epochs, LR=0.0001, warmup=2
  Stage 2: 10 epochs, LR=2e-05, warmup=1
  Total: 37 epochs
  Output: ..\outputs\gfa_v4_8_2


In [3]:
# Cell 3: Import Dataset
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

print("Using GFAMappedDataset with use_autobrep=True")
print("v4.8.2: Only edge_to_faces and bfs_level are required")

Using GFAMappedDataset with use_autobrep=True
v4.8.2: Only edge_to_faces and bfs_level are required


In [4]:
# Cell 4: Load Datasets
gc.collect()
torch.cuda.empty_cache()

print("Loading datasets with AutoBrep topology...")
print("=" * 60)

MAX_TRAIN_SAMPLES = 111000

# Train dataset - LOAD TO RAM
print(f"\n[1/2] Loading TRAIN dataset (max {MAX_TRAIN_SAMPLES:,} samples)...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=True,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
    max_samples=MAX_TRAIN_SAMPLES,
)
print(f"Train: {len(train_dataset):,} samples")

# Val dataset - ON DISK
print("\n[2/2] Loading VAL dataset...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=False,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
)
print(f"Val: {len(val_dataset):,} samples")

print("\n" + "=" * 60)
print("DATASETS READY!")

Loading datasets with AutoBrep topology...

[1/2] Loading TRAIN dataset (max 111,000 samples)...
  Filtered to 132324 samples with AutoBrep data (from 133105)
  Limiting to 111000 samples (from 132324)
  Loading train data to memory (B-Rep + PC + Text)...
    Loading AutoBrep with topology...
    Loading 111000 samples from train_brep_autobrep.h5...
      Done (111000 samples)
    AutoBrep loaded: 9.2GB in 130.1s
    Loading PC (50GB)...
    Loading Text from: c:\Users\User\Desktop\text_splits\train_text_embeddings.h5
    ✓ Text loaded: 174.6GB in 328.5s
  ✓ Loaded 111000 samples: 205.6GB in RAM (B-Rep + PC + Text)
GFAMappedDataset: train with 111000 samples (in memory)
Train: 111,000 samples

[2/2] Loading VAL dataset...
  Filtered to 16535 samples with AutoBrep data (from 16638)
GFAMappedDataset: val with 16535 samples
Val: 16,535 samples

DATASETS READY!


In [22]:
# Cell 5: Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
)

STEPS_PER_EPOCH = len(train_loader)
print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

Train loader: 433 batches
Val loader: 65 batches


In [23]:

# Cell 6: Batch Remapping Function
def remap_batch(batch):
    """Remap collate output keys to model expected keys."""
    remapped = {}
    for k, v in batch.items():
        # Remove 'brep_' prefix for model compatibility
        if k.startswith('brep_'):
            new_key = k[5:]  # Remove 'brep_' prefix
            remapped[new_key] = v
        else:
            remapped[k] = v
    
    # Split pc_features into local and global
    if 'pc_features' in remapped:
        pc = remapped.pop('pc_features')
        remapped['pc_local_features'] = pc[:, :-1, :]   # (B, N, 1024)
        remapped['pc_global_features'] = pc[:, -1, :]   # (B, 1024)
    
    # Rename text keys
    if 'desc_embedding' in remapped:
        remapped['text_features'] = remapped.pop('desc_embedding')
    if 'desc_mask' in remapped:
        remapped['text_mask'] = remapped.pop('desc_mask')
    
    return remapped

# Test remapping and verify topology fields
test_batch = next(iter(train_loader))
test_batch = remap_batch(test_batch)
print("Remapped batch keys:")
for k, v in test_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

# Verify topology fields are present (required for v4.8.2)
assert 'edge_to_faces' in test_batch, "edge_to_faces required!"
assert 'bfs_level' in test_batch, "bfs_level required!"
print("\nTopology fields verified!")

Remapped batch keys:
  has_brep: torch.Size([256])
  has_pointcloud: torch.Size([256])
  has_text: torch.Size([256])
  face_features: torch.Size([256, 192, 48])
  edge_features: torch.Size([256, 512, 12])
  face_mask: torch.Size([256, 192])
  edge_mask: torch.Size([256, 512])
  edge_to_faces: torch.Size([256, 512, 2])
  bfs_level: torch.Size([256, 192])
  face_centroids: torch.Size([256, 192, 3])
  face_normals: torch.Size([256, 192, 3])
  face_areas: torch.Size([256, 192])
  edge_midpoints: torch.Size([256, 512, 3])
  edge_lengths: torch.Size([256, 512])
  idx: torch.Size([256])
  rot_idx: torch.Size([256])
  pc_local_features: torch.Size([256, 47, 1024])
  pc_global_features: torch.Size([256, 1024])
  text_features: torch.Size([256, 256, 3072])
  text_mask: torch.Size([256, 256])

Topology fields verified!


In [24]:
# Cell 7: Create Model (v4.8.2 with optimized config)
gc.collect()
torch.cuda.empty_cache()

from clip4cad.models.clip4cad_gfa_v4_8_2 import CLIP4CAD_GFA_v482, GFAv482Config
import importlib
import clip4cad.losses.gfa_v4_8_2_losses as gfa_losses_module
importlib.reload(gfa_losses_module)

from clip4cad.losses.gfa_v4_8_2_losses import (
    GFAv482LossSmooth,
    compute_modality_gap,
    compute_true_pair_cosine,
    compute_brep_pc_metrics,
    compute_code_diversity,
    compute_active_codes,
    mine_hard_negatives_by_code,
    get_warmup_cosine_scheduler,
)

print("Loss module reloaded!")

# Model configuration (v4.8.2 optimized - uses new defaults)
config = GFAv482Config(
    # Input dimensions (unchanged)
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    # Model dimensions (v4.8.2: +25% capacity)
    d=320,              # Was 256
    d_proj=160,         # Was 128 (maintain ratio)
    # Hierarchical codebook (v4.8.2: more codes)
    n_category=20,      # Was 16
    n_type_per_cat=10,  # Was 8
    n_variant_per_type=4,  # Unchanged
    n_spatial=20,       # Was 16
    code_sparsity=0.1,
    # Architecture (v4.8.2: deeper)
    num_msg_layers=3,   # Unchanged
    num_brep_tf_layers=5,  # Was 4
    num_heads=10,       # Was 8 (keeps head_dim=32)
    dropout=0.1,
)

model = CLIP4CAD_GFA_v482(config).to(device)

# Enable gradient checkpointing for memory efficiency
model.enable_gradient_checkpointing()

print(f"\nModel: CLIP4CAD_GFA_v482 (optimized config)")
print(f"Model parameters: {model.count_parameters():,}")
print(f"Trainable parameters: {model.count_parameters(trainable_only=True):,}")
print(f"Total codes: {model.codebook.total_codes}")
print(f"BRep encoder: {type(model.brep_encoder).__name__}")
print(f"\nConfig summary:")
print(f"  d={config.d}, d_proj={config.d_proj}")
print(f"  Codes: {config.n_category} cat × {config.n_type_per_cat} type × {config.n_variant_per_type} var + {config.n_spatial} spatial")
print(f"  BRep TF layers: {config.num_brep_tf_layers}, heads: {config.num_heads}")

Loss module reloaded!
Enabling gradient checkpointing...
  - BRep transformer: checkpointed
  - Text transformer: checkpointed
  - BRep message layers: checkpointed
Gradient checkpointing enabled (saves ~40% memory)

Model: CLIP4CAD_GFA_v482 (optimized config)
Model parameters: 19,323,422
Trainable parameters: 19,323,422
Total codes: 1040
BRep encoder: SimplifiedTopologyBRepEncoder

Config summary:
  d=320, d_proj=160
  Codes: 20 cat × 10 type × 4 var + 20 spatial
  BRep TF layers: 5, heads: 10


In [25]:
# Cell 8: Loss Function (v4.8.2 Smooth Curriculum)
criterion = GFAv482LossSmooth(
    lambda_recon=0.5,
    lambda_align=0.5,
    lambda_uniform=0.3,
    lambda_code=0.3,
    lambda_diversity=0.1,
    lambda_hard_neg=0.3,
    label_smoothing=0.1,
    tau_start=0.07,
    tau_end=0.05,  # Will actually use 0.06 now
)

# Configure stage epochs
criterion.stage0_epochs = STAGE0_EPOCHS
criterion.stage1_epochs = STAGE1_EPOCHS
criterion.stage2_epochs = STAGE2_EPOCHS

# Scaler for FP16
scaler = GradScaler()

print("Loss function: GFAv482LossSmooth")
print(f"  Label smoothing: {criterion.label_smoothing}")
print(f"  Temperature annealing: {criterion.tau_start} → {criterion.tau_end}")
print(f"  Stage 1 code warmup: {criterion.stage1_code_warmup} epochs")
print(f"  Stage 2 ATP warmup: {criterion.stage2_atp_warmup} epochs")

Loss function: GFAv482LossSmooth
  Label smoothing: 0.1
  Temperature annealing: 0.07 → 0.05
  Stage 1 code warmup: 3 epochs
  Stage 2 ATP warmup: 2 epochs


C:\Users\User\AppData\Local\Temp\ipykernel_12520\3096254577.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [26]:
# Cell 9: Verify Stage 0 Forward Pass
print("Verifying Stage 0 forward pass (BRep-PC only)...")

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast():
        outputs = model.forward_stage0(test_batch)

print("\nStage 0 Outputs:")
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        nan_count = v.isnan().sum().item()
        status = "NaN!" if nan_count > 0 else "OK"
        print(f"  {k}: {v.shape} [{status}]")
    else:
        print(f"  {k}: {v}")

# Test loss with smooth curriculum
loss, loss_dict = criterion(outputs, stage=0, epoch_in_stage=1, global_epoch=1, total_epochs=TOTAL_EPOCHS)
print("\nLoss components (epoch 1):")
for k, v in loss_dict.items():
    print(f"  {k}: {v.item():.4f}")

Verifying Stage 0 forward pass (BRep-PC only)...


C:\Users\User\AppData\Local\Temp\ipykernel_12520\1698504379.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Stage 0 Outputs:
  z_brep: torch.Size([256, 160]) [OK]
  z_pc: torch.Size([256, 160]) [OK]
  z_brep_raw: torch.Size([256, 320]) [OK]
  z_pc_raw: torch.Size([256, 320]) [OK]
  recon: torch.Size([256, 48]) [OK]
  recon_target: torch.Size([256, 48]) [OK]
  tau: torch.Size([]) [OK]

Loss components (epoch 1):
  contrastive: 5.6024
  align: 17.0938
  cosine: 1.0059
  recon: 0.3279
  total: 12.7093


## Stage 0: Anchor BRep to PC

**Goal:** Make BRep encoder produce meaningful features by aligning to pre-trained PC encoder.

**v4.8.2 Changes:**
- Warmup + cosine scheduler (1 epoch warmup)
- Smooth cosine weight decay (1.0 → 0.5)
- Contrastive ramp-up (0.5 → 1.0)
- Label smoothing enabled

**Expected:** BRep-PC cosine ≥ 0.85 by epoch 12

---

### Two options:
1. **Cell 10**: Train Stage 0 from scratch
2. **Cell 10b**: Load existing Stage 0 checkpoint (if already trained)

**Run only ONE of these cells!**

In [10]:
# Cell 10: Stage 0 Training (with warmup scheduler)
# SKIP THIS CELL if loading from checkpoint (run Cell 10b instead)

print("\n" + "="*70)
print("STAGE 0: Anchoring BRep encoder to PC (ShapeLLM anchor)")
print("="*70)

optimizer = AdamW(model.parameters(), lr=STAGE0_LR, weight_decay=WEIGHT_DECAY)
scheduler = get_warmup_cosine_scheduler(
    optimizer,
    warmup_epochs=STAGE0_WARMUP,
    total_epochs=STAGE0_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
)

stage0_history = {
    'loss': [], 'gap': [], 'cosine': [], 'recon': [], 'lr': []
}

global_epoch = 0

for epoch in range(1, STAGE0_EPOCHS + 1):
    global_epoch += 1
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {'gap': [], 'cos': [], 'recon': []}
    
    pbar = tqdm(train_loader, desc=f"Stage 0, Epoch {epoch}/{STAGE0_EPOCHS}")
    
    for batch in pbar:
        batch = remap_batch(batch)
        
        with autocast():
            outputs = model.forward_stage0(batch)
            loss, loss_dict = criterion(
                outputs, stage=0, 
                epoch_in_stage=epoch, 
                global_epoch=global_epoch,
                total_epochs=TOTAL_EPOCHS
            )
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()  # Step per batch for warmup
        
        epoch_loss += loss_dict['total'].item()
        
        with torch.no_grad():
            gap, cos = compute_brep_pc_metrics(outputs['z_brep_raw'], outputs['z_pc_raw'])
            epoch_metrics['gap'].append(gap)
            epoch_metrics['cos'].append(cos)
            epoch_metrics['recon'].append(loss_dict['recon'].item())
        
        current_lr = scheduler.get_last_lr()[0]
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'gap': f"{gap:.3f}",
            'cos': f"{cos:.3f}",
            'lr': f"{current_lr:.2e}",
        })
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_gap = np.mean(epoch_metrics['gap'])
    avg_cos = np.mean(epoch_metrics['cos'])
    avg_recon = np.mean(epoch_metrics['recon'])
    
    stage0_history['loss'].append(avg_loss)
    stage0_history['gap'].append(avg_gap)
    stage0_history['cosine'].append(avg_cos)
    stage0_history['recon'].append(avg_recon)
    stage0_history['lr'].append(current_lr)
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Gap={avg_gap:.4f}, Cos={avg_cos:.4f}, Recon={avg_recon:.4f}, LR={current_lr:.2e}")

# Save Stage 0 checkpoint
torch.save({
    'epoch': STAGE0_EPOCHS,
    'global_epoch': global_epoch,
    'model_state_dict': model.state_dict(),
    'history': stage0_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_stage0.pt')

print("\nStage 0 complete! BRep encoder anchored to PC.")
print(f"Final BRep-PC gap: {stage0_history['gap'][-1]:.4f}")
print(f"Final BRep-PC cosine: {stage0_history['cosine'][-1]:.4f}")


STAGE 0: Anchoring BRep encoder to PC (ShapeLLM anchor)


Stage 0, Epoch 1/12:   0%|          | 0/433 [00:00<?, ?it/s]

C:\Users\User\AppData\Local\Temp\ipykernel_13968\3251577982.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\User\anaconda3\envs\clip4cad\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Epoch 1: Loss=4.4480, Gap=0.4265, Cos=0.7863, Recon=0.1728, LR=3.00e-04


Stage 0, Epoch 2/12:   0%|          | 0/433 [00:00<?, ?it/s]

...

Epoch 10: Loss=4.3167, Gap=0.0890, Cos=0.8695, Recon=0.0330, LR=2.66e-05


Stage 0, Epoch 11/12:   0%|          | 0/433 [00:00<?, ?it/s]

Epoch 11: Loss=4.3799, Gap=0.0846, Cos=0.8702, Recon=0.0324, LR=9.02e-06


Stage 0, Epoch 12/12:   0%|          | 0/433 [00:00<?, ?it/s]

Epoch 12: Loss=4.4784, Gap=0.0811, Cos=0.8706, Recon=0.0322, LR=3.00e-06

Stage 0 complete! BRep encoder anchored to PC.
Final BRep-PC gap: 0.0811
Final BRep-PC cosine: 0.8706


In [27]:
# Cell 10b: Load Stage 0 Checkpoint (ALTERNATIVE to Cell 10)
# Run this cell INSTEAD of Cell 10 if you already trained Stage 0

print("\n" + "="*70)
print("Loading Stage 0 checkpoint...")
print("="*70)

STAGE0_CHECKPOINT = OUTPUT_DIR / 'checkpoint_stage0.pt'

if not STAGE0_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Stage 0 checkpoint not found at {STAGE0_CHECKPOINT}\n"
        f"Either run Cell 10 to train Stage 0, or check the path."
    )

checkpoint = torch.load(STAGE0_CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
stage0_history = checkpoint['history']
global_epoch = checkpoint.get('global_epoch', STAGE0_EPOCHS)

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"  Final BRep-PC gap: {stage0_history['gap'][-1]:.4f}")
print(f"  Final BRep-PC cosine: {stage0_history['cosine'][-1]:.4f}")
print(f"  Final loss: {stage0_history['loss'][-1]:.4f}")

# Verify the loaded model
model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast():
        outputs = model.forward_stage0(test_batch)
    gap, cos = compute_brep_pc_metrics(outputs['z_brep_raw'], outputs['z_pc_raw'])
    print(f"\nVerification on current batch:")
    print(f"  BRep-PC gap: {gap:.4f}")
    print(f"  BRep-PC cosine: {cos:.4f}")

print("\nStage 0 checkpoint loaded successfully!")


Loading Stage 0 checkpoint...
Loaded checkpoint from epoch 12
  Final BRep-PC gap: 0.0811
  Final BRep-PC cosine: 0.8706
  Final loss: 4.4784


C:\Users\User\AppData\Local\Temp\ipykernel_12520\1902337242.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Verification on current batch:
  BRep-PC gap: 0.1814
  BRep-PC cosine: 0.8721

Stage 0 checkpoint loaded successfully!


## Stage 1: Add Text + Codebook

**Goal:** Learn codebook structure and establish relative alignment.

**v4.8.2 Changes:**
- Code loss ramps up over 3 epochs (smooth start)
- Smooth cosine → KL blend for code alignment
- Warmup + cosine scheduler (2 epochs warmup)
- Label smoothing enabled

**Expected:** Diversity > 0.3, text-BRep cosine positive within 5 epochs

In [28]:
# Cell 11: Load Pre-initialized Codebook
print("\n" + "="*70)
print("Loading pre-initialized hierarchical codebook...")
print("="*70)

# Path to pre-computed codebook (generated by scripts/initialize_codebook.py)
CODEBOOK_PATH = OUTPUT_DIR / 'codebook_init.pt'

if not CODEBOOK_PATH.exists():
    raise FileNotFoundError(
        f"Codebook not found at {CODEBOOK_PATH}\n\n"
        f"Generate it first with:\n"
        f"  python scripts/initialize_codebook.py \\\n"
        f"    --text-h5 c:/Users/User/Desktop/text_splits/train_text_embeddings.h5 \\\n"
        f"    --checkpoint {OUTPUT_DIR / 'checkpoint_stage0.pt'} \\\n"
        f"    --output {CODEBOOK_PATH} \\\n"
        f"    --max-samples 75000 \\\n"
        f"    --d 320 --n-category 20 --n-type-per-cat 10 --n-spatial 20"
    )

# Load pre-computed codebook using model's helper method
model.load_codebook(str(CODEBOOK_PATH))

# Reset grounding layers to identity (they weren't trained in Stage 0)
model.reset_grounding_to_identity()

# Show what was loaded
codebook_state = torch.load(CODEBOOK_PATH, map_location='cpu', weights_only=False)
print(f"Codebook loaded from {CODEBOOK_PATH}")
print(f"  Category codes: {codebook_state['category_codes'].shape}")
print(f"  Type codes: {codebook_state['type_codes'].shape}")
print(f"  Variant codes: {codebook_state['variant_codes'].shape}")
print(f"  Spatial codes: {codebook_state['spatial_codes'].shape}")
print(f"  Total codes: {model.codebook.total_codes}")


Loading pre-initialized hierarchical codebook...
Loading pre-initialized codebook from ..\outputs\gfa_v4_8_2\codebook_init.pt...
Loaded codebook with 1040 codes
Resetting grounding layers to identity...
Grounding layers reset to identity (pass-through)
Codebook loaded from ..\outputs\gfa_v4_8_2\codebook_init.pt
  Category codes: torch.Size([20, 320])
  Type codes: torch.Size([20, 10, 320])
  Variant codes: torch.Size([20, 10, 4, 320])
  Spatial codes: torch.Size([20, 320])
  Total codes: 1040


In [19]:
# Cell 12: Verify Stage 1 Forward Pass
print("Verifying Stage 1 forward pass (full with codebook)...")

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast():
        outputs = model(test_batch, stage=1)

print("\nStage 1 Outputs:")
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        nan_count = v.isnan().sum().item()
        status = "NaN!" if nan_count > 0 else "OK"
        print(f"  {k}: {v.shape} [{status}]")
    elif isinstance(v, dict):
        print(f"  {k}: {{...}}")
    else:
        print(f"  {k}: {v}")

# Check active codes
active_codes = compute_active_codes(outputs['w_text'], outputs['w_brep'], outputs['w_pc'])
print("\nActive codes per level (avg):")
for level in ['category', 'type', 'variant', 'spatial']:
    print(f"  {level}: {active_codes[f'{level}_avg']:.1f}")

# Test loss with smooth curriculum
loss, loss_dict = criterion(
    outputs, stage=1, 
    epoch_in_stage=1, 
    global_epoch=STAGE0_EPOCHS + 1,
    total_epochs=TOTAL_EPOCHS
)
print("\nLoss components (epoch 1 with code warmup):")
for k, v in loss_dict.items():
    print(f"  {k}: {v.item():.4f}")

Verifying Stage 1 forward pass (full with codebook)...


C:\Users\User\AppData\Local\Temp\ipykernel_12520\469368637.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Stage 1 Outputs:
  z_text: torch.Size([256, 160]) [OK]
  z_brep: torch.Size([256, 160]) [OK]
  z_pc: torch.Size([256, 160]) [OK]
  z_text_raw: torch.Size([256, 320]) [OK]
  z_brep_raw: torch.Size([256, 320]) [OK]
  z_pc_raw: torch.Size([256, 320]) [OK]
  w_text: {...}
  w_brep: {...}
  w_pc: {...}
  G_text: torch.Size([256, 20, 256]) [OK]
  G_brep: torch.Size([256, 20, 704]) [OK]
  G_pc: torch.Size([256, 20, 48]) [OK]
  n_active: {...}
  level_weights: torch.Size([4]) [OK]
  recon: torch.Size([256, 48]) [OK]
  recon_target: torch.Size([256, 48]) [OK]
  tau: torch.Size([]) [OK]

Active codes per level (avg):
  category: 5.0
  type: 10.0
  variant: 8.0
  spatial: 5.0

Loss components (epoch 1 with code warmup):
  contrastive: 6.0690
  cosine: 0.9473
  code: 1.6730
  diversity: 0.0943
  recon: 39.2933
  total: 8.6581


In [29]:
# Cell 13: Stage 1 Training (with warmup scheduler and smooth curriculum)
print("\n" + "="*70)
print("STAGE 1: Adding text and codebook alignment")
print("="*70)

# Reset optimizer with warmup scheduler
optimizer = AdamW(model.parameters(), lr=STAGE1_LR, weight_decay=WEIGHT_DECAY)
scheduler = get_warmup_cosine_scheduler(
    optimizer,
    warmup_epochs=STAGE1_WARMUP,
    total_epochs=STAGE1_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
)

stage1_history = {
    'loss': [], 'gap_brep': [], 'gap_pc': [],
    'cos_brep': [], 'cos_pc': [], 'diversity': [],
    'contrastive': [], 'code': [], 'lr': [],
}

for epoch in range(1, STAGE1_EPOCHS + 1):
    global_epoch += 1
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {
        'gap_brep': [], 'gap_pc': [],
        'cos_brep': [], 'cos_pc': [],
        'diversity': [], 'contrastive': [], 'code': []
    }
    
    pbar = tqdm(train_loader, desc=f"Stage 1, Epoch {epoch}/{STAGE1_EPOCHS}")
    
    for batch in pbar:
        batch = remap_batch(batch)
        
        with autocast():
            outputs = model(batch, stage=1)
            loss, loss_dict = criterion(
                outputs, stage=1,
                epoch_in_stage=epoch,
                global_epoch=global_epoch,
                total_epochs=TOTAL_EPOCHS
            )
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_loss += loss_dict['total'].item()
        
        with torch.no_grad():
            gap_brep, gap_pc = compute_modality_gap(
                outputs['z_text_raw'], outputs['z_brep_raw'], outputs['z_pc_raw']
            )
            cos_brep, cos_pc = compute_true_pair_cosine(
                outputs['z_text_raw'], outputs['z_brep_raw'], outputs['z_pc_raw']
            )
            diversity = compute_code_diversity(
                outputs['w_text'], outputs['w_brep'], outputs['w_pc'], level='category'
            )
            
            epoch_metrics['gap_brep'].append(gap_brep)
            epoch_metrics['gap_pc'].append(gap_pc)
            epoch_metrics['cos_brep'].append(cos_brep)
            epoch_metrics['cos_pc'].append(cos_pc)
            epoch_metrics['diversity'].append(diversity)
            epoch_metrics['contrastive'].append(loss_dict['contrastive'].item())
            epoch_metrics['code'].append(loss_dict['code'].item())
        
        current_lr = scheduler.get_last_lr()[0]
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'gap': f"{gap_brep:.2f}",
            'cos': f"{cos_brep:.3f}",
            'div': f"{diversity:.3f}",
        })
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    for k in epoch_metrics:
        stage1_history[k].append(np.mean(epoch_metrics[k]))
    stage1_history['loss'].append(avg_loss)
    stage1_history['lr'].append(current_lr)
    
    # Check stage weights for this epoch
    weights = criterion.get_stage_weights(epoch, stage=1)
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Gap(B)={np.mean(epoch_metrics['gap_brep']):.4f}, "
          f"Cos(B)={np.mean(epoch_metrics['cos_brep']):.4f}, Div={np.mean(epoch_metrics['diversity']):.4f}, "
          f"CodeW={weights['code']:.2f}, KLBlend={weights.get('use_kl_blend', 0):.2f}")

# Save Stage 1 checkpoint
torch.save({
    'epoch': STAGE0_EPOCHS + STAGE1_EPOCHS,
    'global_epoch': global_epoch,
    'model_state_dict': model.state_dict(),
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_stage1.pt')

print("\nStage 1 complete! Text and codebook aligned.")


STAGE 1: Adding text and codebook alignment


Stage 1, Epoch 1/15:   0%|          | 0/433 [00:00<?, ?it/s]

C:\Users\User\AppData\Local\Temp\ipykernel_12520\165002311.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1: Loss=5.5875, Gap(B)=8.6105, Cos(B)=0.7891, Div=0.8111, CodeW=0.10, KLBlend=0.03


Stage 1, Epoch 2/15:   0%|          | 0/433 [00:00<?, ?it/s]

...

Epoch 13: Loss=4.7300, Gap(B)=1.7458, Cos(B)=0.9847, Div=0.5598, CodeW=0.30, KLBlend=0.40


Stage 1, Epoch 14/15:   0%|          | 0/433 [00:00<?, ?it/s]

Epoch 14: Loss=4.7220, Gap(B)=1.6751, Cos(B)=0.9851, Div=0.5604, CodeW=0.30, KLBlend=0.40


Stage 1, Epoch 15/15:   0%|          | 0/433 [00:00<?, ?it/s]

Epoch 15: Loss=4.7167, Gap(B)=1.6533, Cos(B)=0.9853, Div=0.5603, CodeW=0.30, KLBlend=0.40

Stage 1 complete! Text and codebook aligned.


## Stage 2: Gap Closing + Hard Negatives

**Goal:** Close the absolute modality gap and improve fine-grained discrimination.

**v4.8.2 Changes:**
- ATP/CU/hard_neg ramp up over 2 epochs (smooth start)
- Adaptive hard negative boost (1.1-1.5x based on similarity)
- Warmup + cosine scheduler (1 epoch warmup)
- Label smoothing enabled

**Expected:** Gap decreases toward 0, no instability at start

In [ ]:
# Cell 14: Mine Hard Negatives
print("\n" + "="*70)
print("Mining hard negatives before Stage 2...")
print("="*70)

hard_negatives = mine_hard_negatives_by_code(
    model, train_loader, device, 
    top_k=10, max_batches=50,
    remap_fn=remap_batch
)
print(f"Mined {len(hard_negatives)} hard negative sets")

In [ ]:
# Cell 15: Stage 2 Training (with warmup and smooth ramp-up)
print("\n" + "="*70)
print("STAGE 2: Gap closing and hard negative mining")
print("="*70)

# Reset optimizer with warmup scheduler
optimizer = AdamW(model.parameters(), lr=STAGE2_LR, weight_decay=WEIGHT_DECAY)
scheduler = get_warmup_cosine_scheduler(
    optimizer,
    warmup_epochs=STAGE2_WARMUP,
    total_epochs=STAGE2_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
)

stage2_history = {
    'loss': [], 'gap_brep': [], 'gap_pc': [],
    'cos_brep': [], 'cos_pc': [], 'diversity': [],
    'contrastive': [], 'align': [], 'uniform': [], 'code': [], 'lr': [],
}

best_gap = float('inf')

for epoch in range(1, STAGE2_EPOCHS + 1):
    global_epoch += 1
    
    # Re-mine hard negatives every 3 epochs
    if epoch > 1 and epoch % 3 == 0:
        print("Re-mining hard negatives...")
        hard_negatives = mine_hard_negatives_by_code(
            model, train_loader, device, 
            top_k=10, max_batches=50,
            remap_fn=remap_batch
        )
    
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {
        'gap_brep': [], 'gap_pc': [],
        'cos_brep': [], 'cos_pc': [],
        'diversity': [], 'contrastive': [], 'align': [], 'uniform': [], 'code': []
    }
    
    pbar = tqdm(train_loader, desc=f"Stage 2, Epoch {epoch}/{STAGE2_EPOCHS}")
    
    for batch_idx, batch in enumerate(pbar):
        batch = remap_batch(batch)
        
        # Get hard negatives for this batch
        batch_size = batch['face_features'].shape[0]
        start_idx = batch_idx * batch_size
        batch_hard_negs = [
            hard_negatives.get(start_idx + i) for i in range(batch_size)
        ]
        
        with autocast():
            outputs = model(batch, stage=2)
            loss, loss_dict = criterion(
                outputs, stage=2,
                epoch_in_stage=epoch,
                global_epoch=global_epoch,
                total_epochs=TOTAL_EPOCHS,
                hard_negatives=batch_hard_negs
            )
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_loss += loss_dict['total'].item()
        
        with torch.no_grad():
            gap_brep, gap_pc = compute_modality_gap(
                outputs['z_text_raw'], outputs['z_brep_raw'], outputs['z_pc_raw']
            )
            cos_brep, cos_pc = compute_true_pair_cosine(
                outputs['z_text_raw'], outputs['z_brep_raw'], outputs['z_pc_raw']
            )
            diversity = compute_code_diversity(
                outputs['w_text'], outputs['w_brep'], outputs['w_pc'], level='category'
            )
            
            epoch_metrics['gap_brep'].append(gap_brep)
            epoch_metrics['gap_pc'].append(gap_pc)
            epoch_metrics['cos_brep'].append(cos_brep)
            epoch_metrics['cos_pc'].append(cos_pc)
            epoch_metrics['diversity'].append(diversity)
            epoch_metrics['contrastive'].append(loss_dict['contrastive'].item())
            epoch_metrics['align'].append(loss_dict['align'].item())
            epoch_metrics['uniform'].append(loss_dict['uniform'].item())
            epoch_metrics['code'].append(loss_dict['code'].item())
        
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'gap': f"{gap_brep:.2f}",
            'cos': f"{cos_brep:.3f}",
        })
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_gap = np.mean(epoch_metrics['gap_brep'])
    current_lr = scheduler.get_last_lr()[0]
    
    for k in epoch_metrics:
        stage2_history[k].append(np.mean(epoch_metrics[k]))
    stage2_history['loss'].append(avg_loss)
    stage2_history['lr'].append(current_lr)
    
    # Check stage weights for this epoch
    weights = criterion.get_stage_weights(epoch, stage=2)
    
    if avg_gap < best_gap:
        best_gap = avg_gap
        torch.save({
            'epoch': STAGE0_EPOCHS + STAGE1_EPOCHS + epoch,
            'global_epoch': global_epoch,
            'model_state_dict': model.state_dict(),
            'best_gap': best_gap,
            'config': config,
        }, OUTPUT_DIR / 'checkpoint_best.pt')
        print(f"  New best gap: {best_gap:.4f}!")
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Gap(B)={np.mean(epoch_metrics['gap_brep']):.4f}, "
          f"Cos(B)={np.mean(epoch_metrics['cos_brep']):.4f}, Align={np.mean(epoch_metrics['align']):.4f}, "
          f"ATPW={weights['atp']:.2f}, CUW={weights['uniform']:.2f}")

# Save Stage 2 checkpoint (end of stage)
torch.save({
    'epoch': STAGE0_EPOCHS + STAGE1_EPOCHS + STAGE2_EPOCHS,
    'global_epoch': global_epoch,
    'model_state_dict': model.state_dict(),
    'best_gap': best_gap,
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'stage2_history': stage2_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_stage2.pt')

print("\n" + "="*70)
print("Training Complete!")
print(f"Best gap: {best_gap:.4f}")
print(f"Stage 2 checkpoint saved to {OUTPUT_DIR / 'checkpoint_stage2.pt'}")

In [ ]:
# Cell 16: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Stage 0: BRep-PC metrics
ax = axes[0, 0]
ax.plot(stage0_history['gap'], label='BRep-PC Gap')
ax.plot(stage0_history['cosine'], label='BRep-PC Cosine')
ax.axhline(y=0.85, color='g', linestyle='--', alpha=0.5, label='Target Cosine (0.85)')
ax.set_xlabel('Epoch')
ax.set_title('Stage 0: BRep-PC Anchoring')
ax.legend()
ax.grid(True, alpha=0.3)

# Stage 1+2: Modality Gap
ax = axes[0, 1]
s1_epochs = list(range(1, len(stage1_history['gap_brep']) + 1))
s2_epochs = list(range(len(s1_epochs) + 1, len(s1_epochs) + len(stage2_history['gap_brep']) + 1))
ax.plot(s1_epochs, stage1_history['gap_brep'], label='Stage 1 - BRep')
ax.plot(s2_epochs, stage2_history['gap_brep'], label='Stage 2 - BRep')
ax.axhline(y=0, color='g', linestyle='--', alpha=0.5, label='Target (0)')
ax.axvline(x=len(s1_epochs), color='r', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Gap')
ax.set_title('Modality Gap (should → 0)')
ax.legend()
ax.grid(True, alpha=0.3)

# True-Pair Cosine
ax = axes[0, 2]
ax.plot(s1_epochs, stage1_history['cos_brep'], label='Stage 1')
ax.plot(s2_epochs, stage2_history['cos_brep'], label='Stage 2')
ax.axhline(y=1.0, color='g', linestyle='--', alpha=0.5, label='Target (1.0)')
ax.axvline(x=len(s1_epochs), color='r', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cosine Similarity')
ax.set_title('True-Pair Cosine (should → 1.0)')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss
ax = axes[1, 0]
all_loss = stage0_history['loss'] + stage1_history['loss'] + stage2_history['loss']
ax.plot(all_loss)
ax.axvline(x=STAGE0_EPOCHS, color='r', linestyle='--', alpha=0.3, label='Stage 1')
ax.axvline(x=STAGE0_EPOCHS + STAGE1_EPOCHS, color='r', linestyle='--', alpha=0.3, label='Stage 2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.grid(True, alpha=0.3)

# Diversity
ax = axes[1, 1]
all_div = stage1_history['diversity'] + stage2_history['diversity']
ax.plot(all_div)
ax.axhline(y=0.3, color='g', linestyle='--', alpha=0.5, label='Min Target (>0.3)')
ax.set_xlabel('Epoch (Stage 1+2)')
ax.set_ylabel('Diversity')
ax.set_title('Code Diversity')
ax.legend()
ax.grid(True, alpha=0.3)

# Summary
axes[1, 2].axis('off')
summary_text = (
    f"GFA v4.8.2 Training Summary\n"
    f"(Optimized Config)\n\n"
    f"Model: d={config.d}, {model.count_parameters():,} params\n"
    f"Codes: {model.codebook.total_codes}\n\n"
    f"Stage 0 (Anchor): {STAGE0_EPOCHS} epochs\n"
    f"  BRep-PC Cosine: {stage0_history['cosine'][-1]:.4f}\n\n"
    f"Stage 1 (Align): {STAGE1_EPOCHS} epochs\n"
    f"  Gap: {stage1_history['gap_brep'][-1]:.4f}\n\n"
    f"Stage 2 (Close): {STAGE2_EPOCHS} epochs\n"
    f"  Best Gap: {best_gap:.4f}\n"
    f"  Final Cosine: {stage2_history['cos_brep'][-1]:.4f}\n"
    f"  Diversity: {stage2_history['diversity'][-1]:.4f}"
)
axes[1, 2].text(0.5, 0.5, summary_text, ha='center', va='center',
                fontsize=11, family='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

In [ ]:
# Cell 17: Save Final Model
torch.save({
    'config': config,
    'model_state_dict': model.state_dict(),
    'best_gap': best_gap,
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'stage2_history': stage2_history,
}, OUTPUT_DIR / 'gfa_v4_8_2_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'gfa_v4_8_2_final.pt'}")
print(f"Best modality gap: {best_gap:.4f}")
print(f"Model parameters: {model.count_parameters():,}")
print(f"Total codes: {model.codebook.total_codes}")

In [ ]:
# Cell 18: Retrieval Evaluation
print("Testing retrieval performance...")

model.eval()
all_z_text = []
all_z_brep = []
all_z_pc = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Encoding"):
        batch = remap_batch(batch)
        with autocast():
            outputs = model(batch, stage=2)
        
        all_z_text.append(outputs['z_text'].cpu())
        all_z_brep.append(outputs['z_brep'].cpu())
        all_z_pc.append(outputs['z_pc'].cpu())

z_text = torch.cat(all_z_text, dim=0)
z_brep = torch.cat(all_z_brep, dim=0)
z_pc = torch.cat(all_z_pc, dim=0)

# Normalize
z_text = F.normalize(z_text, dim=-1)
z_brep = F.normalize(z_brep, dim=-1)
z_pc = F.normalize(z_pc, dim=-1)

N = z_text.shape[0]
print(f"\nEvaluation set: {N} samples")

# Text -> BRep retrieval
sim_t2b = z_text @ z_brep.T
ranks_t2b = (sim_t2b.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1_t2b = (ranks_t2b < 1).float().mean().item() * 100
r5_t2b = (ranks_t2b < 5).float().mean().item() * 100
r10_t2b = (ranks_t2b < 10).float().mean().item() * 100

# Text -> PC retrieval
sim_t2p = z_text @ z_pc.T
ranks_t2p = (sim_t2p.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1_t2p = (ranks_t2p < 1).float().mean().item() * 100
r5_t2p = (ranks_t2p < 5).float().mean().item() * 100

# BRep -> PC retrieval (geometry-only)
sim_b2p = z_brep @ z_pc.T
ranks_b2p = (sim_b2p.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1_b2p = (ranks_b2p < 1).float().mean().item() * 100
r5_b2p = (ranks_b2p < 5).float().mean().item() * 100

print("\n" + "="*50)
print("Retrieval Results (v4.8.2):")
print("="*50)
print(f"Text -> BRep: R@1={r1_t2b:.1f}%, R@5={r5_t2b:.1f}%, R@10={r10_t2b:.1f}%")
print(f"Text -> PC:   R@1={r1_t2p:.1f}%, R@5={r5_t2p:.1f}%")
print(f"BRep -> PC:   R@1={r1_b2p:.1f}%, R@5={r5_b2p:.1f}%")
print("="*50)